# Vì sao chọn ORB? — Đối sánh ORB / SIFT / AKAZE

Notebook trả lời **một** câu hỏi: vì sao pipeline mặc định dùng **ORB** thay vì SIFT/AKAZE.

Nhìn ảnh khớp đặc trưng thì cả ba bộ dò gần như giống nhau — khác biệt thật nằm ở **thời gian**
và **sai số hình học**. Notebook này biến khác biệt đó thành số và biểu đồ.

**Nguyên tắc**

- Số liệu chính thức đọc từ `experiments/summary.csv` + `experiments/per_pair.csv`
  (do `experiments/exp_feature_matching.py` sinh ra trên 693 cặp), không lấy từ output cell.
- Notebook chỉ **đọc** `data/`; biểu đồ ghi ra thư mục tạm, không ghi đè CSV nào.
- ⚠️ `summary.csv` hiện chỉ có **ORB và SIFT**. Số AKAZE full-693 nằm ở
  `PROGRESS_REPORT.md` §2.1 (lần chạy riêng, chưa lưu per-pair). Mục 5 chạy AKAZE trên
  5 cặp để so cùng điều kiện; nếu muốn CSV đủ 3 bộ dò thì chạy lại
  `exp_feature_matching.py --detectors ORB SIFT AKAZE`.
- Recall/Precision/F1 trong CSV đo bằng default **cũ** (`floor 30 / open 3`).
  Các chỉ số *căn chỉnh* (thời gian, inliers, reprojection, corner error) không phụ thuộc default đó.
- Trước khi chụp ảnh báo cáo: **Kernel → Restart & Run All**.


In [ ]:

from pathlib import Path
import csv
import os
import sys
import tempfile
import time

# Tìm project root (thư mục chứa dataset_utils.py) để notebook chạy được từ mọi nơi.
PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "dataset_utils.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "experiments"))

import cv2
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

%matplotlib inline

CHART_DIR = Path(tempfile.gettempdir()) / "pcb_notebook_charts"
CHART_DIR.mkdir(parents=True, exist_ok=True)
plt.rcParams["figure.dpi"] = 110

from dataset_utils import find_pairs, load_image
import module_preprocess as mp
import module_visualization as mv
import exp_feature_matching as efm

DETECTOR_COLORS = {"ORB": "#4C72B0", "SIFT": "#DD8452", "AKAZE": "#55A868"}
print("Project root :", PROJECT_ROOT)
print("Chart dir    :", CHART_DIR)


def to_display(image, max_width: int = 1000):
    """BGR/gray -> RGB và thu nhỏ để hiển thị."""
    if image.ndim == 2:
        image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
    else:
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    height, width = image.shape[:2]
    if width > max_width:
        scale = max_width / width
        image = cv2.resize(image, (max_width, int(height * scale)), interpolation=cv2.INTER_AREA)
    return image


def show_stack(items, figsize=(14, 4.2)):
    fig, axes = plt.subplots(len(items), 1, figsize=(figsize[0], figsize[1] * len(items)),
                             squeeze=False)
    for row, (image, title) in enumerate(items):
        axes[row, 0].imshow(to_display(image))
        axes[row, 0].set_title(title, fontsize=10)
        axes[row, 0].axis("off")
    plt.tight_layout()
    plt.show()


## 1. Dữ liệu

`data/PKU-Market-PCB/pairs_manifest.csv` mô tả 693 cặp (template = bo mạch sạch, test = có lỗi).
Mỗi cặp chỉ chứa **một** loại khuyết tật, nhãn nằm trong `labels/`.


In [ ]:
pairs = find_pairs(str(PROJECT_ROOT / "data"))
print(f"Tìm thấy {len(pairs)} cặp ảnh")

sample = next((p for p in pairs if p["pair_id"] == "01_short_01"), pairs[0])
template = load_image(sample["template_path"])
test = load_image(sample["test_path"])
print("Cặp mẫu      :", sample["pair_id"], "| lớp:", sample["class_name"])
print("Kích thước   :", "template", template.shape, "| test", test.shape)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].imshow(to_display(template))
axes[0].set_title("Template (bo mạch sạch)")
axes[1].imshow(to_display(test))
axes[1].set_title(f"Test có lỗi: {sample['pair_id']}")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()


## 2. Cùng một cặp ảnh, ba bộ dò

Mỗi bộ dò trích keypoint → ghép cặp → RANSAC để loại ngoại lai → ước lượng Homography.
Số trên banner là inliers, sai số tái chiếu và thời gian của chính lần chạy này.


In [ ]:
DEMO_DETECTORS = ["ORB", "SIFT", "AKAZE"]
demo_align = {}

for name in DEMO_DETECTORS:
    start = time.perf_counter()
    try:
        aligned, stats = mp.align_images_with_stats(template, test, detector=name, min_ncc=None)
    except mp.AlignmentError as exc:
        aligned, stats = test, dict(exc.stats)
    demo_align[name] = (aligned, stats)
    elapsed = (time.perf_counter() - start) * 1000
    reproj = stats.get("reprojection_error", float("nan"))
    print(f"{name:<6} {elapsed:7.1f} ms | inliers {stats.get('inliers', 0):5d} | "
          f"reproj {reproj:.2f} px | thành công: {stats.get('success')}")


In [ ]:
match_items = []
for name in DEMO_DETECTORS:
    aligned, stats = demo_align[name]
    if stats.get("inlier_pts_test") is not None:
        match_items.append((mv.draw_feature_matches(test, template, stats), f"{name}: inlier matches"))
show_stack(match_items, figsize=(14, 3.0))


## 3. Nhìn kỹ hơn: độ khít sau khi nắn ảnh

Bàn cờ (trái) và alpha blend (phải) của **cùng một cặp** với ba bộ dò.
Đường mạch chạy liền qua ranh giới ô cờ = căn chỉnh khít. Lệch rõ nhất ở mép ảnh.


In [ ]:
board_items = []
for name in DEMO_DETECTORS:
    aligned, stats = demo_align[name]
    verification = mv.draw_homography_verification(template, aligned, mode="both")
    board_items.append((verification, f"{name}: checkerboard + alpha blend"))
show_stack(board_items, figsize=(14, 2.4))


## 4. Số liệu chính thức trên 693 cặp

Đọc trực tiếp `experiments/summary.csv`. `corner error` = sai số vị trí 4 góc ảnh sau khi nắn,
so với phép nắn giả lập đã biết trước (đo trên ảnh bị nắn lệch ±3°, ±40 px, ±3%).


In [ ]:
summary_path = PROJECT_ROOT / "experiments" / "summary.csv"
with summary_path.open("r", encoding="utf-8-sig", newline="") as handle:
    summary_rows = [r for r in csv.DictReader(handle) if r["kind"] == "original"]

header = (f"{'detector':<8}{'align ms':>10}{'inliers':>9}{'reproj px':>11}"
          f"{'corner med':>12}{'recall':>9}{'precision':>11}{'F1':>8}")
print(header)
print("-" * len(header))
for row in summary_rows:
    print(f"{row['detector']:<8}{float(row['time_align_ms']):>10.1f}{float(row['inliers']):>9.0f}"
          f"{float(row['reprojection_error']):>11.3f}{float(row['corner_error_median']):>12.3f}"
          f"{float(row['recall']):>9.4f}{float(row['precision']):>11.4f}{float(row['f1']):>8.4f}")


In [ ]:
per_pair_path = PROJECT_ROOT / "experiments" / "per_pair.csv"
with per_pair_path.open("r", encoding="utf-8-sig", newline="") as handle:
    per_pair_rows = list(csv.DictReader(handle))

print("Phân bố corner error (p95 = 5% cặp tệ nhất):")
for name in DETECTOR_COLORS:
    values = [float(r["corner_error"]) for r in per_pair_rows
              if r["detector"] == name and r["kind"] == "original"
              and r["corner_error"] not in ("", "nan")]
    if values:
        print(f"  {name:<6} n={len(values):4d} | trung vị {np.median(values):.3f} px | "
              f"p95 {np.percentile(values, 95):.3f} px")
    else:
        print(f"  {name:<6} không có trong per_pair.csv — xem PROGRESS_REPORT.md §2.1")


## 5. Chạy thử 5 cặp cho đủ ba bộ dò

Phần này đo **tại chỗ** để AKAZE có số cùng điều kiện với ORB/SIFT. Chỉ đo bước căn chỉnh
(không nắn lệch giả lập, không phát hiện lỗi), nên số ở đây là "demo 5 cặp", không thay CSV 693 cặp.


In [ ]:
DEMO_PAIRS = 5
live = {name: {"ms": [], "inliers": []} for name in DEMO_DETECTORS}

for pair in pairs[:DEMO_PAIRS]:
    tpl = load_image(pair["template_path"])
    tst = load_image(pair["test_path"])
    for name in DEMO_DETECTORS:
        start = time.perf_counter()
        try:
            _, stats = mp.align_images_with_stats(tpl, tst, detector=name, min_ncc=None)
        except mp.AlignmentError as exc:
            stats = dict(exc.stats)
        live[name]["ms"].append((time.perf_counter() - start) * 1000)
        live[name]["inliers"].append(stats.get("inliers", 0) or 0)

print(f"{'detector':<8}{'ms/cặp (trung vị)':>20}{'inliers (trung vị)':>20}")
for name in DEMO_DETECTORS:
    print(f"{name:<8}{np.median(live[name]['ms']):>20.1f}{np.median(live[name]['inliers']):>20.0f}")


## 6. Biểu đồ đánh đổi: nhanh — chính xác — chất lượng cuối

Ba trục này không cùng chiều. Câu hỏi đúng không phải "bộ dò nào tốt nhất" mà
"đánh đổi nào hợp với camera công nghiệp".


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
names = [r["detector"] for r in summary_rows]
colors = [DETECTOR_COLORS.get(n, "gray") for n in names]

axes[0].barh(names, [float(r["time_align_ms"]) for r in summary_rows], color=colors)
axes[0].set_xscale("log")
axes[0].invert_yaxis()
axes[0].set_xlabel("ms/cặp (thang log)")
axes[0].set_title("Tốc độ: ORB nhanh hơn nhiều lần")

axes[1].barh(names, [float(r["corner_error_median"]) for r in summary_rows], color=colors)
axes[1].invert_yaxis()
axes[1].set_xlabel("corner error trung vị (px)")
axes[1].set_title("Hình học: SIFT/AKAZE chính xác hơn")

f1_values = [float(r["f1"]) for r in summary_rows]
axes[2].barh(names, f1_values, color=colors)
axes[2].invert_yaxis()
axes[2].set_xlim(min(f1_values) - 0.01, max(f1_values) + 0.005)
axes[2].set_xlabel("F1 phát hiện lỗi (693 cặp)")
axes[2].set_title("Chất lượng cuối: gần như nhau")

for ax in axes:
    ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
chart_path = CHART_DIR / "feature_matching_tradeoff.png"
plt.savefig(chart_path, dpi=130, bbox_inches="tight")
plt.show()
print("Đã lưu biểu đồ:", chart_path)


## 7. Kết luận rút ra từ chính CSV

Không viết tay kết luận: đoạn dưới tự tính từ `summary.csv`.


In [ ]:
best_f1 = max(float(r["f1"]) for r in summary_rows)
slowest = max(float(r["time_align_ms"]) for r in summary_rows)
fastest = min(summary_rows, key=lambda r: float(r["time_align_ms"]))
close_to_best = [r["detector"] for r in summary_rows if float(r["f1"]) >= best_f1 - 0.01]

ratio = slowest / float(fastest["time_align_ms"])
gap = best_f1 - float(fastest["f1"])
print(f"F1 cao nhất                : {best_f1:.4f}")
print(f"Detector nhanh nhất        : {fastest['detector']} ({float(fastest['time_align_ms']):.0f} ms/cặp)")
print(f"Các detector trong 0.01 F1 : {close_to_best}")
print(f"Chênh F1 tối đa            : {gap:.4f}")
print(f"Chênh thời gian            : {ratio:.1f} lần (chậm nhất / nhanh nhất)")
print()
print("→ F1 gần như bằng nhau nhưng chênh thời gian cả bậc, nên chọn ORB làm mặc định.")
print("→ SIFT/AKAZE chỉ dùng khi cần sai số hình học cực thấp (ví dụ ảnh chụp lệch nhiều).")


---

**Ghi chú khi commit**

- Notebook không được import bởi `main.py`/pipeline; chỉ để trình bày.
- Xoá output trước khi commit:

```powershell
jupyter nbconvert --clear-output --inplace experiments/notebooks/feature_matching.ipynb
```

- Không sửa `exp_feature_matching.py`, `data/`, `results/` từ notebook.
